# Climate Comfort Index · Dummy Regressor

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts CCI +1, +2 and +3 days ahead*

This notebook trains and evaluates the **Dummy Regressor** model for the **Climate Comfort Index (CCI)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

`DummyRegressor(strategy="mean")` ignores the features and predicts the training-set average CCI
for every day and every horizon. It is the correct **null model**: every other CCI experiment,
and both naive baselines, should beat it.

The hypothesis is that **the constant mean is the weakest forecast in the project**, and that
the gap between it and the two naive baselines shows how much current weather (persistence) and
the calendar (climatology) are each worth before any model is fitted.

This notebook is **not a deployment candidate**; it establishes the floor.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `DummyRegressor(strategy="mean")` | sklearn's standard regression null model |
| Features | Received, then ignored | The constant does not depend on `X` |
| Scaling | None | A constant is unaffected by feature scale |
| Strategy | One model per horizon (the endpoint returns three values) | Matches the deployed artefact layout |
| Selection | None: `default_params` is used as is | A null model has nothing worth tuning (section 5) |
| Objective | validation **mean absolute error (MAE)**, in CCI points (lower is better) | Used for every comparison |

### Workflow

All ten experiment notebooks run the same code from `nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model specification in section 1 differs between notebooks**, and because this is a null model the tuning steps of the other notebooks are left out: the final model uses `default_params` directly.

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Feature-family ablation | What each block of engineered features contributes |
| 5 | Fixed configuration | Use `default_params` as is: no search |
| 6 | Final model | Final model against the baselines |
| 7 | Feature importance | Which features the model relies on |
| 8 | Generalisation check | Training vs validation performance |
| 9 | Save artefacts | Model bundle + `/model-metadata` document |
| 10 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyRegressor

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "comfort_climate"
EXPERIMENT = "cci_dummy_mean"
ALGORITHM = "Dummy Regressor"

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It creates the experiment from the estimator class, which the package builds (seeding it and, where requested, scaling the features), so no helper function is needed. It also sets the configuration the model always uses (`default_params`).

The model has a single setting, `strategy="mean"`, which is used as is. The dummy regressor has
no `random_state`, so no seed is passed.

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    estimator=DummyRegressor,
    default_params={"strategy": "mean"},  # used as is: no search in this notebook
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is
reserved for a single evaluation in the comparison notebook, once one model has been
selected, so nothing in this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up
> filter and an embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Two naive forecasts, both fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Persistence** | Tomorrow's CCI equals today's CCI | The current weather, but not the season |
| **Climatology** | The average CCI for that day of the year | The season, but not the current weather |

A model must beat **both** to be worth deploying: that shows it has combined current conditions
with the seasonal cycle.

In [ ]:
exp.run_baselines()

---
## 4. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column), and the model is refitted at each step. This measures what each block of engineering contributes instead of assuming it. `ablation_params` are used throughout so the differences come from the features, not from tuning. Scores are on validation (MAE).

> A null model ignores its inputs, so every row should score the same: this confirms the model does not use `X`.

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 5. Fixed configuration

No hyperparameter search is run. The final model uses `default_params` exactly
as defined in section 1. A null model has no settings whose choice would be
informative, and keeping it fixed means the floor it sets is a property of the
labels, not of a search.

In [ ]:
exp.use_default_params()

---
## 6. Final model

The model is fitted with `default_params` and compared with every baseline (it appears as `default`). `gain_of_final_model` is positive when the final model is better than that row.

`pred_std` is shown next to the error metrics. A spread far below the observed spread means the
model is playing safe by predicting close to the average, which can give a respectable MAE while
missing exactly the uncomfortable days users care about. Expect error to grow with the horizon.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 7. Feature importance

A dummy model ignores the features, so there is no importance to report.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature that looks unimportant on its own may belong to an essential block. The ablation in section 4 is the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 8. Generalisation check

Training and validation scores side by side. A small gap means the model
generalises; a large gap means it has memorised patterns in the training years that
do not carry over to 2024.

In [ ]:
exp.generalisation()

---
## 9. Save artefacts

The fitted model(s) are written to
`models/comfort_climate/experiments/cci_dummy_mean/` with:

| File | Contents |
|:--|:--|
| `cci_dummy_mean_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array
> at prediction time and would silently give wrong answers if the columns were
> permuted.

In [ ]:
exp.export()

---
## 10. Conclusions

- **This is the floor, not the bar.** Persistence and climatology both use information and should
  beat it comfortably; the size of those gaps (section 6) measures what current weather and
  the calendar are each worth.
- **`pred_std` is zero by construction.** This model *is* the "retreat to the average" that
  `pred_std` detects in the other notebooks, so it calibrates how to read that number.
- **Mean, not median.** The mean minimises squared error, while MAE is minimised by the median.
  Because CCI is skewed by poor-comfort days, a median would score slightly better on MAE; it is
  not used, so the floor stays the simplest possible forecast.
- **Ablation.** Every feature set scores identically (section 4), confirming the model
  ignores `X`.

> The test split has still not been used. It is scored exactly once, in the `cci_comparison` notebook, after one model has been selected on validation performance across all five CCI experiments.